# Chapter 14 — Build It Yourself: SFT & LoRA

Watch a base model *ignore* an instruction, build the **loss mask** that trains only the response,
implement **LoRA** from scratch, then supervised-fine-tune a tiny GPT and watch it start answering.

> 💡 **No GPU needed.** Running a ✍️ cell before you fill it prints a friendly ⚠️, not an error.

### Setup — run this (the model & chat helpers; you don't edit it)

In [ ]:
import math, torch, torch.nn as nn, torch.nn.functional as F

PAIRS = [('hi','hello!'), ('bye','goodbye!'), ('thanks','welcome!'), ('yes','great!'), ('no','okay.')]
U, A, E = '\x01', '\x02', '\x03'                 # <user>, <assistant>, <end> markers
texts = [U+i+A+r+E for i,r in PAIRS]
chars = sorted(set(''.join(texts))); stoi = {c:i for i,c in enumerate(chars)}; itos = {i:c for c,i in stoi.items()}
V = len(chars); BLK = max(len(t) for t in texts) + 16

def make_example(instr, resp):               # (x, y) with the PROMPT masked to -100
    seq = U+instr+A+resp+E; ids=[stoi[c] for c in seq]; x,y = ids[:-1], ids[1:]
    ap = seq.index(A)
    return x, [(t if p>=ap else -100) for p,t in enumerate(y)]

class GPT(nn.Module):
    def __init__(s, V, BLK, C=64, NH=4, NL=2):
        super().__init__(); s.NH,s.HD=NH,C//NH; s.tok=nn.Embedding(V,C); s.pos=nn.Embedding(BLK,C)
        s.bl=nn.ModuleList([_Blk(C,NH) for _ in range(NL)]); s.lf=nn.LayerNorm(C); s.hd=nn.Linear(C,V)
    def forward(s, idx):
        x=s.tok(idx)+s.pos(torch.arange(idx.size(1)))
        for b in s.bl: x=b(x)
        return s.hd(s.lf(x))
class _Blk(nn.Module):
    def __init__(s,C,NH): super().__init__(); s.l1,s.l2=nn.LayerNorm(C),nn.LayerNorm(C); s.at=_Att(C,NH); s.ff=nn.Sequential(nn.Linear(C,4*C),nn.GELU(),nn.Linear(4*C,C))
    def forward(s,x): x=x+s.at(s.l1(x)); return x+s.ff(s.l2(x))
class _Att(nn.Module):
    def __init__(s,C,NH): super().__init__(); s.NH,s.HD=NH,C//NH; s.q,s.k,s.v,s.proj=(nn.Linear(C,C) for _ in range(4))
    def forward(s,x):
        B,T,C=x.shape; q,k,v=(l(x).view(B,T,s.NH,s.HD).transpose(1,2) for l in (s.q,s.k,s.v))
        m=torch.triu(torch.ones(T,T)*float('-inf'),1)
        return s.proj((torch.softmax((q@k.transpose(-2,-1))/math.sqrt(s.HD)+m,-1)@v).transpose(1,2).reshape(B,T,C))

@torch.no_grad()
def respond(model, instr, max_new=12):       # greedy generate the response to an instruction
    ids=[stoi[c] for c in U+instr+A]
    for _ in range(max_new):
        n=model(torch.tensor(ids).unsqueeze(0))[0,-1].argmax().item()
        if itos[n]==E: break
        ids.append(n)
    return ''.join(itos[t] for t in ids[ids.index(stoi[A])+1:])
print(f'{len(PAIRS)} chat pairs | vocab {V} | block {BLK} — model & helpers ready')

## Step 1 — A base model doesn't answer  ▶️ Run this

An untrained model given an instruction just babbles — it has no idea it's supposed to *respond*.
That's the behaviour SFT will install.

In [ ]:
torch.manual_seed(0)
base_model = GPT(V, BLK)
for i, r in PAIRS[:3]:
    print(f"  '{i}' -> {respond(base_model, i)!r}   (babble)")

## Step 2 — Build the loss mask  ✍️ Your turn

Each example is the whole conversation `<user> hi <assistant> hello! <end>`, but we train only on
the **response**. Build `y` so prompt-token targets are `-100` (ignored) and response tokens keep
their real value.

<details><summary>Stuck? Show the answer</summary>

```python
y_masked = [(t if p >= a_pos else -100) for p, t in enumerate(y)]
```
</details>

In [ ]:
seq = U + 'hi' + A + 'hello!' + E
ids = [stoi[c] for c in seq]
x, y = ids[:-1], ids[1:]
a_pos = seq.index(A)                    # response tokens are the ones after <assistant>

# ✍️ mask the prompt: target -100 for positions before the response, else the real token
y_masked = None      # replace

shown = None if y_masked is None else ''.join('_' if t==-100 else ('<end>' if itos[t]==E else itos[t]) for t in y_masked)
print('trained target:', shown, "  ('_' = masked prompt)")

In [ ]:
# ▶️ Check your work
try:
    if y_masked is None:
        print('⚠️  Fill in the ✍️ cell above (build y_masked), then re-run. (Expected until you do.)')
    elif y_masked == make_example('hi', 'hello!')[1]:
        print(f'✅ Correct — only the response is trained: {shown!r}. The prompt tokens are -100 (ignored).')
    else:
        print('⚠️  Not quite — target should be -100 for p < a_pos and the real token for p >= a_pos.')
except Exception as e:
    print('❌', e)

## Step 3 — Implement LoRA  ✍️ Your turn

LoRA freezes the weight `W` and adds a trainable low-rank update `B·A`. Implement the forward
(`W·x + B·A·x`) and watch it fit a low-rank target with a handful of parameters.

<details><summary>Stuck? Show the answer</summary>

```python
return self.base(x) + (x @ self.A.T) @ self.B.T
```
</details>

In [ ]:
class LoRALinear(nn.Module):
    def __init__(s, base, r=8):
        super().__init__(); s.base = base
        for p in base.parameters(): p.requires_grad = False        # freeze W
        s.A = nn.Parameter(torch.randn(r, base.in_features) * 0.02)
        s.B = nn.Parameter(torch.zeros(base.out_features, r))       # start as a no-op
    def forward(s, x):
        # ✍️ return W·x + B·A·x
        return None      # replace

torch.manual_seed(0)
base = nn.Linear(64, 64, bias=False)
W_target = base.weight.data + (torch.randn(64, 8) @ torch.randn(8, 64)) * 0.1   # a rank-8 change
X = torch.randn(256, 64); Y = X @ W_target.T
lora = LoRALinear(base, 8)
final = None
if lora(X) is not None:                                             # forward implemented?
    opt = torch.optim.Adam([lora.A, lora.B], lr=0.02)
    for _ in range(300):
        loss = ((lora(X) - Y) ** 2).mean(); opt.zero_grad(); loss.backward(); opt.step()
    final = loss.item()
print('fit loss:', final, '| trainable params:', sum(p.numel() for p in lora.parameters() if p.requires_grad))

In [ ]:
# ▶️ Check your work
try:
    if final is None:
        print('⚠️  Fill in the ✍️ forward (W·x + B·A·x), then re-run. (Expected until you do.)')
    elif final < 0.01:
        print(f'✅ LoRA fit the rank-8 target to {final:.5f} with just 1,024 trainable params — the low-rank bet.')
    else:
        print(f'⚠️  Fit loss {final:.3f} is high — check the forward is base(x) + (x @ A.T) @ B.T.')
except Exception as e:
    print('❌', e)

## Step 4 — Supervised fine-tune  ▶️ Run this

Now SFT the base model from Step 1 on the masked chat examples, and watch it turn from babble into
an instruction-follower. (Just run it — this uses the loss mask and the model you already have.)

In [ ]:
torch.manual_seed(0)
model = GPT(V, BLK)
examples = [make_example(i, r) for i, r in PAIRS]
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
for _ in range(400):
    loss = sum(F.cross_entropy(model(torch.tensor(x).unsqueeze(0)).view(-1, V), torch.tensor(y), ignore_index=-100)
               for x, y in examples) / len(examples)
    opt.zero_grad(); loss.backward(); opt.step()

model.eval()
print(f'after SFT (loss {loss.item():.4f}):')
correct = sum(respond(model, i) == r for i, r in PAIRS)
for i, r in PAIRS:
    got = respond(model, i); print(f"  '{i}' -> {got!r}   {'✓' if got==r else '✗'}")
print(f'{correct}/{len(PAIRS)} correct — the base model became an instruction-follower.')

## 🎉 You built SFT + LoRA.

You saw the base model babble, masked the loss to the response, implemented LoRA, and fine-tuned a
GPT into an instruction-follower. This is how every chat model is made. Take it to your **GPT** in
the [mini-project](../project/) (SFT *with* LoRA), then on to
[Chapter 15 — RL alignment](../../15-finetuning-rl/). 🎓